# Praktikum & Latihan Mandiri Pertemuan 4 — Analisis Data Penjualan Retail

**Mata Kuliah** : Data Science  
**Pertemuan** : 4  
**Semester / SKS** : 7 (Tujuh) / 3 SKS  
**Dosen Pengampu** : Samsul Bahri, S.Kom, M.Kom  
**Institusi** : Fakultas Ilmu Komputer — Institut Teknologi Tangerang Selatan (ITTS)

---
### Mahasiswa
**Nama** : Akmal Yusuf Abiyoga  
**NIM** : 1001230016  
**Program Studi** : Sistem Informasi

---
### Tujuan Praktikum
1. Membuat dan mengimpor dataset penjualan.
2. Memeriksa tipe data dan kualitas data.
3. Melakukan *filtering*, *sorting*, dan transformasi kolom.
4. Menghitung statistik ringkas.
5. Menyimpan hasil analisis ke CSV, Excel, dan JSON.

### Persiapan Environment — Instalasi Package
Menjalankan instalasi pustaka yang dibutuhkan (`pandas`, `numpy`, dan `openpyxl` untuk ekspor ke format Excel `.xlsx`).

In [11]:
!pip install pandas numpy openpyxl

## Bagian 1: Praktikum Analisis Data Penjualan Retail

### Langkah 1 — Membuat Dataset & Menyimpan Dataset Awal
Dataset terdiri atas 8 transaksi penjualan perangkat komputer dan aksesori pada periode `2026-09-01` hingga `2026-09-04`.

In [12]:
import pandas as pd
import numpy as np

data = {
    "id_transaksi": [
        "T001", "T002", "T003", "T004",
        "T005", "T006", "T007", "T008"
    ],
    "tanggal": [
        "2026-09-01", "2026-09-01",
        "2026-09-02", "2026-09-02",
        "2026-09-03", "2026-09-03",
        "2026-09-04", "2026-09-04"
    ],
    "produk": [
        "Laptop", "Mouse", "Keyboard", "Monitor",
        "Mouse", "Laptop", "Keyboard", "Monitor"
    ],
    "kategori": [
        "Elektronik", "Aksesori", "Aksesori", "Elektronik",
        "Aksesori", "Elektronik", "Aksesori", "Elektronik"
    ],
    "jumlah": [2, 5, 3, 2, 4, 1, 6, 3],
    "harga": [
        7500000, 150000, 350000, 1800000,
        150000, 7500000, 350000, 1800000
    ]
}

df = pd.DataFrame(data)
print(df)

# Simpan dataset awal ke CSV
df.to_csv("penjualan.csv", index=False)

  id_transaksi     tanggal    produk    kategori  jumlah    harga
0         T001  2026-09-01    Laptop  Elektronik       2  7500000
1         T002  2026-09-01     Mouse    Aksesori       5   150000
2         T003  2026-09-02  Keyboard    Aksesori       3   350000
3         T004  2026-09-02   Monitor  Elektronik       2  1800000
4         T005  2026-09-03     Mouse    Aksesori       4   150000
5         T006  2026-09-03    Laptop  Elektronik       1  7500000
6         T007  2026-09-04  Keyboard    Aksesori       6   350000
7         T008  2026-09-04   Monitor  Elektronik       3  1800000


### Langkah 2 — Memeriksa Kualitas Data
Memeriksa dimensi tabel (`shape`), tipe data tiap kolom (`dtypes`), informasi memori (`info()`), jumlah nilai kosong (`isna().sum()`), serta baris duplikat (`duplicated().sum()`).

In [13]:
print("Ukuran data:", df.shape)
print("\nTipe data:")
print(df.dtypes)

print("\nInformasi data:")
df.info()

print("\nNilai kosong:")
print(df.isna().sum())

print("\nDuplikasi (seluruh kolom):")
print(df.duplicated().sum())

print("\nDuplikasi berdasarkan ID transaksi:")
print(df["id_transaksi"].duplicated().sum())

Ukuran data: (8, 6)

Tipe data:
id_transaksi      str
tanggal           str
produk            str
kategori          str
jumlah          int64
harga           int64
dtype: object

Informasi data:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_transaksi  8 non-null      str  
 1   tanggal       8 non-null      str  
 2   produk        8 non-null      str  
 3   kategori      8 non-null      str  
 4   jumlah        8 non-null      int64
 5   harga         8 non-null      int64
dtypes: int64(2), str(4)
memory usage: 516.0 bytes

Nilai kosong:
id_transaksi    0
tanggal         0
produk          0
kategori        0
jumlah          0
harga           0
dtype: int64

Duplikasi (seluruh kolom):
0

Duplikasi berdasarkan ID transaksi:
0


**Interpretasi Kualitas Data:**
- Dataset memiliki **8 baris** dan **6 kolom** (`shape = (8, 6)`).
- Tidak ditemukan nilai kosong (*missing value* = `0` pada seluruh kolom) maupun duplikasi transaksi (`0`).
- Kolom `tanggal` masih bertipe teks (`object`/`str`), sehingga perlu dikonversi ke `datetime64`.

### Langkah 3 — Memperbaiki Tipe Data
Mengubah kolom `tanggal` menjadi tipe `datetime` agar dapat dianalisis berdasarkan periode waktu, serta memastikan `jumlah` dan `harga` bertipe numerik.

In [14]:
df["tanggal"] = pd.to_datetime(
    df["tanggal"],
    errors="coerce"
)

df["jumlah"] = pd.to_numeric(
    df["jumlah"],
    errors="coerce"
)

df["harga"] = pd.to_numeric(
    df["harga"],
    errors="coerce"
)

print(df.dtypes)

id_transaksi               str
tanggal         datetime64[us]
produk                     str
kategori                   str
jumlah                   int64
harga                    int64
dtype: object


### Langkah 4 — Menambahkan Kolom Turunan
Menghitung omzet per transaksi dengan rumus:
$$\text{Omzet} = \text{Jumlah} \times \text{Harga}$$
Serta menambahkan kolom `bulan` untuk kebutuhan agregasi periode bulanan.

In [15]:
df["total_penjualan"] = df["jumlah"] * df["harga"]

print(
    df[[
        "id_transaksi", "produk",
        "jumlah", "harga", "total_penjualan"
    ]]
)

# Tambahkan kolom bulan untuk analisis periode
df["bulan"] = df["tanggal"].dt.to_period("M").astype(str)
print("\nData dengan kolom bulan:")
print(df[["id_transaksi", "tanggal", "bulan", "total_penjualan"]])

  id_transaksi    produk  jumlah    harga  total_penjualan
0         T001    Laptop       2  7500000         15000000
1         T002     Mouse       5   150000           750000
2         T003  Keyboard       3   350000          1050000
3         T004   Monitor       2  1800000          3600000
4         T005     Mouse       4   150000           600000
5         T006    Laptop       1  7500000          7500000
6         T007  Keyboard       6   350000          2100000
7         T008   Monitor       3  1800000          5400000

Data dengan kolom bulan:
  id_transaksi    tanggal    bulan  total_penjualan
0         T001 2026-09-01  2026-09         15000000
1         T002 2026-09-01  2026-09           750000
2         T003 2026-09-02  2026-09          1050000
3         T004 2026-09-02  2026-09          3600000
4         T005 2026-09-03  2026-09           600000
5         T006 2026-09-03  2026-09          7500000
6         T007 2026-09-04  2026-09          2100000
7         T008 2026-09-04  

### Langkah 5 — Memfilter dan Mengurutkan Data
- **Filtering**: Memilih transaksi dengan omzet minimal Rp1.000.000.
- **Sorting**: Mengurutkan transaksi dari omzet tertinggi ke terendah.

In [16]:
# Cari transaksi dengan omzet minimal Rp1.000.000
transaksi_besar = df[
    df["total_penjualan"] >= 1_000_000
]
print("Transaksi dengan omzet >= Rp1.000.000:")
print(transaksi_besar)

# Urutkan transaksi berdasarkan omzet tertinggi
peringkat = df.sort_values(
    "total_penjualan",
    ascending=False
)
print("\nUrutan transaksi berdasarkan omzet tertinggi:")
print(
    peringkat[
        ["produk", "jumlah", "total_penjualan"]
    ]
)

Transaksi dengan omzet >= Rp1.000.000:
  id_transaksi    tanggal    produk    kategori  jumlah    harga  \
0         T001 2026-09-01    Laptop  Elektronik       2  7500000   
2         T003 2026-09-02  Keyboard    Aksesori       3   350000   
3         T004 2026-09-02   Monitor  Elektronik       2  1800000   
5         T006 2026-09-03    Laptop  Elektronik       1  7500000   
6         T007 2026-09-04  Keyboard    Aksesori       6   350000   
7         T008 2026-09-04   Monitor  Elektronik       3  1800000   

   total_penjualan    bulan  
0         15000000  2026-09  
2          1050000  2026-09  
3          3600000  2026-09  
5          7500000  2026-09  
6          2100000  2026-09  
7          5400000  2026-09  

Urutan transaksi berdasarkan omzet tertinggi:
     produk  jumlah  total_penjualan
0    Laptop       2         15000000
5    Laptop       1          7500000
7   Monitor       3          5400000
3   Monitor       2          3600000
6  Keyboard       6          2100000
2  Ke

### Langkah 6 — Analisis Ringkas per Produk
Menggunakan `groupby("produk")` untuk menghitung total unit terjual, total omzet, rata-rata harga, dan jumlah transaksi per produk.

In [17]:
ringkasan_produk = (
    df.groupby("produk")
    .agg(
        total_unit=("jumlah", "sum"),
        total_omzet=("total_penjualan", "sum"),
        rata_harga=("harga", "mean"),
        jumlah_transaksi=("id_transaksi", "count")
    )
    .reset_index()
    .sort_values("total_omzet", ascending=False)
)

print(ringkasan_produk)

     produk  total_unit  total_omzet  rata_harga  jumlah_transaksi
1    Laptop           3     22500000   7500000.0                 2
2   Monitor           5      9000000   1800000.0                 2
0  Keyboard           9      3150000    350000.0                 2
3     Mouse           9      1350000    150000.0                 2


**Interpretasi:**
`Laptop` menghasilkan omzet terbesar (**Rp22.500.000**) meskipun total unit yang terjual hanya **3 unit**, jauh lebih sedikit dibandingkan `Mouse` (9 unit) atau `Keyboard` (9 unit). Hal ini menunjukkan bahwa volume penjualan unit tidak selalu berbanding lurus dengan kontribusi omzet.

### Langkah 7 — Menghitung Statistik Deskriptif
Menampilkan ringkasan statistik (`count`, `mean`, `std`, `min`, kuartil `25%`, `50%`, `75%`, dan `max`) serta total omzet keseluruhan.

In [18]:
print(df[["jumlah", "harga", "total_penjualan"]].describe())

total_omzet = df["total_penjualan"].sum()
print(f"\nTotal omzet: Rp{total_omzet:,.0f}")

         jumlah         harga  total_penjualan
count  8.000000  8.000000e+00     8.000000e+00
mean   3.250000  2.450000e+06     4.500000e+06
std    1.669046  3.190387e+06     4.896136e+06
min    1.000000  1.500000e+05     6.000000e+05
25%    2.000000  3.000000e+05     9.750000e+05
50%    3.000000  1.075000e+06     2.850000e+06
75%    4.250000  3.225000e+06     5.925000e+06
max    6.000000  7.500000e+06     1.500000e+07

Total omzet: Rp36,000,000


### Langkah 8 — Ekspor Hasil Analisis
Menyimpan tabel `ringkasan_produk` ke dalam tiga format: CSV, Excel (`.xlsx`), dan JSON.

In [19]:
# CSV
ringkasan_produk.to_csv(
    "ringkasan_produk.csv",
    index=False
)

# Excel
ringkasan_produk.to_excel(
    "ringkasan_produk.xlsx",
    index=False
)

# JSON
ringkasan_produk.to_json(
    "ringkasan_produk.json",
    orient="records",
    indent=2
)

print("Ketiga file (CSV, Excel, JSON) berhasil diekspor!")

Ketiga file (CSV, Excel, JSON) berhasil diekspor!


---
## Bagian 2: Praktikum Lanjutan — Dataset yang Memiliki Masalah (`data_kotor`)

Pada bagian ini kita melakukan pembersihan data secara bertahap:
1. Merapikan nama kolom menjadi huruf kecil (`lowercase`) dan menggunakan *underscore* (`_`).
2. Mengubah kolom `jumlah` menjadi tipe numerik.
3. Mengidentifikasi jumlah nilai kosong (*missing value*).
4. Mengisi nilai kosong dengan median `jumlah`.
5. Menghitung `total_penjualan`.
6. Mengidentifikasi kemungkinan transaksi duplikat berdasarkan aturan bisnis.

In [20]:
data_kotor = pd.DataFrame({
    " Nama Produk ": ["Laptop", "Mouse", "Keyboard", "Mouse"],
    "Jumlah": ["2", "5", None, "3"],
    "Harga": [7500000, 150000, 350000, 150000],
    "Kategori": ["Elektronik", "Aksesori", "Aksesori", "Aksesori"]
})

print("Data sebelum dibersihkan:")
print(data_kotor)

# 1. Rapikan nama kolom
data_kotor.columns = (
    data_kotor.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# 2. Konversi tipe data
data_kotor["jumlah"] = pd.to_numeric(
    data_kotor["jumlah"],
    errors="coerce"
)

# 3. Periksa missing value
print("\nJumlah missing value per kolom:")
print(data_kotor.isna().sum())

# 4. Imputasi median
median_jumlah = data_kotor["jumlah"].median()
data_kotor["jumlah"] = data_kotor["jumlah"].fillna(
    median_jumlah
)

# 5. Hitung omzet
data_kotor["total_penjualan"] = (
    data_kotor["jumlah"] * data_kotor["harga"]
)

print("\nData setelah dibersihkan:")
print(data_kotor)

# 6. Identifikasi kemungkinan transaksi duplikat berdasarkan aturan bisnis (produk, harga, kategori sama)
print("\nBaris dengan produk & harga yang berulang:")
print(data_kotor[data_kotor.duplicated(subset=["nama_produk", "harga", "kategori"], keep=False)])

Data sebelum dibersihkan:
   Nama Produk  Jumlah    Harga    Kategori
0        Laptop      2  7500000  Elektronik
1         Mouse      5   150000    Aksesori
2      Keyboard    NaN   350000    Aksesori
3         Mouse      3   150000    Aksesori

Jumlah missing value per kolom:
nama_produk    0
jumlah         1
harga          0
kategori       0
dtype: int64

Data setelah dibersihkan:
  nama_produk  jumlah    harga    kategori  total_penjualan
0      Laptop     2.0  7500000  Elektronik       15000000.0
1       Mouse     5.0   150000    Aksesori         750000.0
2    Keyboard     3.0   350000    Aksesori        1050000.0
3       Mouse     3.0   150000    Aksesori         450000.0

Baris dengan produk & harga yang berulang:
  nama_produk  jumlah   harga  kategori  total_penjualan
1       Mouse     5.0  150000  Aksesori         750000.0
3       Mouse     3.0  150000  Aksesori         450000.0


> **Catatan Penting:** Mengisi jumlah barang yang hilang dengan median (`3.0`) hanyalah latihan teknis. Dalam transaksi riil, jumlah barang yang tidak diketahui harus diverifikasi ke dokumen sumber (faktur/kasir) karena imputasi dapat mengubah nilai penjualan aktual.

---
## Bagian 3: Soal Latihan Mandiri (Nomor 14)

Menggunakan DataFrame `df` dari praktikum penjualan utama.

### Soal 1: Tampilkan hanya kolom `produk` dan `harga`

In [21]:
soal_1 = df[["produk", "harga"]]
print(soal_1)

     produk    harga
0    Laptop  7500000
1     Mouse   150000
2  Keyboard   350000
3   Monitor  1800000
4     Mouse   150000
5    Laptop  7500000
6  Keyboard   350000
7   Monitor  1800000


### Soal 2: Tampilkan transaksi dengan jumlah barang lebih dari 3

In [22]:
soal_2 = df[df["jumlah"] > 3]
print(soal_2)

  id_transaksi    tanggal    produk  kategori  jumlah   harga  \
1         T002 2026-09-01     Mouse  Aksesori       5  150000   
4         T005 2026-09-03     Mouse  Aksesori       4  150000   
6         T007 2026-09-04  Keyboard  Aksesori       6  350000   

   total_penjualan    bulan  
1           750000  2026-09  
4           600000  2026-09  
6          2100000  2026-09  


### Soal 3: Urutkan data dari total penjualan terkecil ke terbesar

In [23]:
soal_3 = df.sort_values("total_penjualan", ascending=True)
print(soal_3)

  id_transaksi    tanggal    produk    kategori  jumlah    harga  \
4         T005 2026-09-03     Mouse    Aksesori       4   150000   
1         T002 2026-09-01     Mouse    Aksesori       5   150000   
2         T003 2026-09-02  Keyboard    Aksesori       3   350000   
6         T007 2026-09-04  Keyboard    Aksesori       6   350000   
3         T004 2026-09-02   Monitor  Elektronik       2  1800000   
7         T008 2026-09-04   Monitor  Elektronik       3  1800000   
5         T006 2026-09-03    Laptop  Elektronik       1  7500000   
0         T001 2026-09-01    Laptop  Elektronik       2  7500000   

   total_penjualan    bulan  
4           600000  2026-09  
1           750000  2026-09  
2          1050000  2026-09  
6          2100000  2026-09  
3          3600000  2026-09  
7          5400000  2026-09  
5          7500000  2026-09  
0         15000000  2026-09  


### Soal 4: Hitung rata-rata harga produk

In [24]:
rata_rata_harga = df["harga"].mean()
print(f"Rata-rata harga produk (seluruh transaksi): Rp{rata_rata_harga:,.0f}")

print("\nHarga rata-rata per jenis produk:")
print(df.groupby("produk")["harga"].mean())

Rata-rata harga produk (seluruh transaksi): Rp2,450,000

Harga rata-rata per jenis produk:
produk
Keyboard     350000.0
Laptop      7500000.0
Monitor     1800000.0
Mouse        150000.0
Name: harga, dtype: float64


### Soal 5: Hitung total unit yang terjual

In [25]:
total_unit = df["jumlah"].sum()
print(f"Total unit yang terjual: {total_unit} unit")

Total unit yang terjual: 26 unit


### Soal 6: Tampilkan transaksi kategori `Aksesori`

In [26]:
transaksi_aksesori = df[df["kategori"] == "Aksesori"]
print(transaksi_aksesori)

  id_transaksi    tanggal    produk  kategori  jumlah   harga  \
1         T002 2026-09-01     Mouse  Aksesori       5  150000   
2         T003 2026-09-02  Keyboard  Aksesori       3  350000   
4         T005 2026-09-03     Mouse  Aksesori       4  150000   
6         T007 2026-09-04  Keyboard  Aksesori       6  350000   

   total_penjualan    bulan  
1           750000  2026-09  
2          1050000  2026-09  
4           600000  2026-09  
6          2100000  2026-09  


### Soal 7: Tambahkan kolom pajak simulasi sebesar 11% dari total penjualan

In [27]:
df["pajak"] = df["total_penjualan"] * 0.11
print(df[["id_transaksi", "produk", "total_penjualan", "pajak"]])

  id_transaksi    produk  total_penjualan      pajak
0         T001    Laptop         15000000  1650000.0
1         T002     Mouse           750000    82500.0
2         T003  Keyboard          1050000   115500.0
3         T004   Monitor          3600000   396000.0
4         T005     Mouse           600000    66000.0
5         T006    Laptop          7500000   825000.0
6         T007  Keyboard          2100000   231000.0
7         T008   Monitor          5400000   594000.0


### Soal 8: Hitung omzet per kategori

In [28]:
omzet_kategori = (
    df.groupby("kategori")
    .agg(
        total_unit=("jumlah", "sum"),
        total_omzet=("total_penjualan", "sum"),
        total_pajak=("pajak", "sum")
    )
    .reset_index()
    .sort_values("total_omzet", ascending=False)
)
print(omzet_kategori)

     kategori  total_unit  total_omzet  total_pajak
1  Elektronik           8     31500000    3465000.0
0    Aksesori          18      4500000     495000.0


### Soal 9: Ubah nama kolom `produk` menjadi `nama_produk`

In [29]:
df = df.rename(columns={"produk": "nama_produk"})
print(df)

  id_transaksi    tanggal nama_produk    kategori  jumlah    harga  \
0         T001 2026-09-01      Laptop  Elektronik       2  7500000   
1         T002 2026-09-01       Mouse    Aksesori       5   150000   
2         T003 2026-09-02    Keyboard    Aksesori       3   350000   
3         T004 2026-09-02     Monitor  Elektronik       2  1800000   
4         T005 2026-09-03       Mouse    Aksesori       4   150000   
5         T006 2026-09-03      Laptop  Elektronik       1  7500000   
6         T007 2026-09-04    Keyboard    Aksesori       6   350000   
7         T008 2026-09-04     Monitor  Elektronik       3  1800000   

   total_penjualan    bulan      pajak  
0         15000000  2026-09  1650000.0  
1           750000  2026-09    82500.0  
2          1050000  2026-09   115500.0  
3          3600000  2026-09   396000.0  
4           600000  2026-09    66000.0  
5          7500000  2026-09   825000.0  
6          2100000  2026-09   231000.0  
7          5400000  2026-09   594000.0  


### Soal 10: Simpan DataFrame ke CSV tanpa indeks

In [30]:
df.to_csv("latihan_mandiri.csv", index=False)
print("DataFrame berhasil disimpan ke 'latihan_mandiri.csv' tanpa indeks!")

DataFrame berhasil disimpan ke 'latihan_mandiri.csv' tanpa indeks!
